In [1]:
import torch
print(torch.__version__)

2.11.0+cu128


In [2]:
if torch.cuda.is_available():
    print("GPU is available!")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available. Using CPU")

GPU is available!
Using GPU: Tesla T4


* EX-1 -> y = x^2
* EX-2 -> y = x^2 , z = sin(y)
* EX-3 -> z = w*x+b ,y = sigmoid(z)
* Ex-4 -> vector x =[1.0,2.0,3.0] ,y= (x**2).mean()

# Ex-1

In [3]:
x = torch.tensor(3.0, requires_grad=True) 

# requires_grad=True -> by default it's False , when we need to calculate gradient we need to make True 
x

tensor(3., requires_grad=True)

`requires_grad=True` : Tells PyTorch to track all operations on this tensor for the computation graph.

In [4]:
y = x ** 2
y

tensor(9., grad_fn=<PowBackward0>)

`grad_fn=<PowBackward0>`: Automatically created by PyTorch during the forward pass to know how to apply the chain rule during backpropagation.

In [5]:
y.backward()

`y.backward()`: Computes the derivative of y with respect to x.

In [6]:
x.grad

tensor(6.)

`x.grad`: Stores the numerical derivative value dy_dx = 2x = 2 times 3.0 = 6.0

# EX-2

In [7]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2
z = torch.sin(y)

In [8]:
z.backward()

In [9]:
x.grad

tensor(-5.4668)

# EX-3

Manual way

In [10]:
x = torch.tensor(6.7) # input feature
y = torch.tensor(0.0) # output feature

w = torch.tensor(1.0) # weight
b = torch.tensor(0.0) # bias



In [11]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction,target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1-epsilon)
    return -(target * torch.log(prediction) + (1 - target)* torch.log(1-prediction))

In [12]:
# forward pass
z = w * x + b 
y_pred = torch.sigmoid(z)

loss = binary_cross_entropy_loss(y_pred,y)

In [13]:
loss

tensor(6.7012)

In [14]:
# Derivatives

# 1. dl/d(y_pred) : loss with respect to prediction
dloss_by_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))

# 2. dy_pred/dz : Prediction with respect to z (sigmoid derivatives)
dy_pred_by_dz = y_pred * (1 - y_pred)

# 3.  dz/dw and dz/db : z with respect to w and b
dz_by_dw = x 
dz_by_db = 1

dl_by_dw = dloss_by_dy_pred * dy_pred_by_dz * dz_by_dw
dl_by_db = dloss_by_dy_pred * dy_pred_by_dz * dz_by_db

In [15]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dl_by_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dl_by_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


Autograd

In [16]:
x = torch.tensor(6.7)
y = torch.tensor(0.0) # we dont use requires_grad=True b's  we dont need to calculate gradient with respect x and y

w = torch.tensor(1.0,requires_grad=True)
b = torch.tensor(0.0,requires_grad=True)

In [17]:
z = w * x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [18]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [19]:
loss = binary_cross_entropy_loss(y_pred,y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

### computation graph
```mermaid
graph LR
    %% Inputs and Leaf Nodes
    w[w] --> Mult((Multiply))
    x[x] --> Mult
    b[b] --> Add((Add))
    y[y] --> LossFunc((Loss Function))

    %% Intermediate Operations and Tensors
    Mult --> wx[w * x]
    wx --> Add
    Add --> z[z]
    z --> Sig((Sigmoid))
    Sig --> y_pred[y_pred]

    %% Final Output
    y_pred --> LossFunc
    LossFunc --> Loss[Loss]

    %% Styling for better visualization
    style w fill:#000,stroke:#333,stroke-width:2px
    style b fill:#000,stroke:#333,stroke-width:2px
    style x fill:#000,stroke:#333,stroke-width:1px
    style y fill:#000,stroke:#333,stroke-width:1px
    style Loss fill:#000,stroke:#333,stroke-width:2px
```


In [20]:
loss.backward()

In [21]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


# EX-4 

In [22]:
x = torch.tensor([1.0,2.0,3.0],requires_grad=True)

y = (x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [23]:
y.backward()

In [24]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

# Ex-5

In [25]:
# 1. Inputs & Parameters (Vectors of size 3)
w = torch.tensor([2.0, -1.0, 0.5], requires_grad=True)
b = torch.tensor(-1.0, requires_grad=True)
x = torch.tensor([0.5, 1.5, -2.0])  # Features
y = torch.tensor(1.0)                # Label

# 2. Forward Pass (w · x + b)
z = torch.dot(w, x) + b
y_pred = torch.sigmoid(z)

# 3. Loss Calculation
loss = binary_cross_entropy_loss(y_pred, y)

# 4. Compute Gradients
loss.backward()

print(f"w.grad: {w.grad}")  # Will output a gradient vector with 3 elements [dw1, dw2, dw3]


w.grad: tensor([-0.4621, -1.3862,  1.8483])


### computation graph
```mermaid
graph LR
    %% Inputs for index 1
    w1[w1] --> Mult1((Multiply))
    x1[x1] --> Mult1
    Mult1 --> wx1[w1 * x1] --> SumAll((Sum Components))

    %% Inputs for index 2
    w2[w2] --> Mult2((Multiply))
    x2[x2] --> Mult2
    Mult2 --> wx2[w2 * x2] --> SumAll

    %% Inputs for index 3
    w3[w3] --> Mult3((Multiply))
    x3[x3] --> Mult3
    Mult3 --> wx3[w3 * x3] --> SumAll

    %% Bias addition
    SumAll --> wx_dot[w · x]
    wx_dot --> Add((Add))
    b[b] --> Add

    %% Activation and Loss
    Add --> z[z]
    z --> Sig((Sigmoid))
    Sig --> y_pred[y_pred]
    y_pred --> LossFunc((Loss Function))
    y[y] --> LossFunc
    LossFunc --> Loss[Loss]
```


# Clearing grad

In [26]:
x = torch.tensor(3.0, requires_grad=True)

# PASS 1
y1 = x ** 2
y1.backward()
print(f"Pass 1 gradient (expected 2*3 = 6): {x.grad}") # Output: 6.0

# PASS 2 (Without clearing)
y2 = x ** 2
y2.backward()
print(f"Pass 2 gradient (expected 6, but accumulated): {x.grad}") # Output: 12.0 (6 + 6)

# PASS 3 (Without clearing)
y3 = x ** 2
y3.backward()
print(f"Pass 3 gradient (expected 6, but accumulated): {x.grad}") # Output: 18.0 (12 + 6)


Pass 1 gradient (expected 2*3 = 6): 6.0
Pass 2 gradient (expected 6, but accumulated): 12.0
Pass 3 gradient (expected 6, but accumulated): 18.0


In [27]:
x = torch.tensor(3.0, requires_grad=True)

# PASS 1
y1 = x ** 2
y1.backward()
print(f"Pass 1 gradient (expected 2*3 = 6): {x.grad}") 
x.grad.zero_()

# PASS 2 (With clearing)
y2 = x ** 2
y2.backward()
print(f"Pass 2 gradient (expected 6, but accumulated): {x.grad}")
x.grad.zero_()

# PASS 3 (With clearing)
y3 = x ** 2
y3.backward()
print(f"Pass 3 gradient (expected 6, but accumulated): {x.grad}") 
# x.grad.zero_()


Pass 1 gradient (expected 2*3 = 6): 6.0
Pass 2 gradient (expected 6, but accumulated): 6.0
Pass 3 gradient (expected 6, but accumulated): 6.0


* **The Rule:** PyTorch always **adds** new gradients to old ones; it never deletes them automatically.
* **The Problem:** If you run `.backward()` multiple times, your gradients will keep adding up and give wrong results.
* **The Fix:** Run **`x.grad.zero_()`** to wipe the gradient back to zero before your next calculation.


# Disable gradient tracking

* **Saves Memory and Speed:** Turning off gradient tracking stops PyTorch from building a computation graph, making your code run much faster and use less RAM.
* **Used for Testing (Inference):** You only need gradients during training to update weights; you do not need them when testing or using the model to make predictions.


* Option 1 - requires_grad_(False)
* Option 2 - detach()
* Option 3 - torch.no_grad()

### Option 1

In [36]:
x = torch.tensor(3.0, requires_grad=True)
x

tensor(3., requires_grad=True)

In [37]:
y = x ** 2
y.backward()
x.grad

tensor(6.)

In [38]:
x.requires_grad_(False)
x

tensor(3.)

In [39]:
y = x ** 2
y.backward()
x.grad

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

### Option 2

In [40]:
x = torch.tensor(3.0,requires_grad=True)
x

tensor(3., requires_grad=True)

In [41]:
z = x.detach()
z

tensor(3.)

In [42]:
y = x ** 2
y

tensor(9., grad_fn=<PowBackward0>)

In [43]:
y1 = z ** 2
y1

tensor(9.)

In [44]:
y1.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

### Option 3

In [45]:
x = torch.tensor(3.0,requires_grad=True)
x

tensor(3., requires_grad=True)

In [46]:
with torch.no_grad():
    y = x ** 2

y

tensor(9.)

In [47]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn